## Producer A — Kafka Event Stream Initialisation

This cell initialises and runs **Producer A**, which reads camera events from `camera_event_A.csv`
and publishes them to the Kafka topic `camera-events-A` in batches grouped by `batch_id`.

### Key Parameters

| Parameter | Value | Rationale |
|---|---|---|
| `bootstrap_servers` | `kafka:9092` | Kafka broker address as defined in Docker Compose network. Using the service name `kafka` instead of `localhost` ensures correct container-to-container routing. |
| `topic` | `camera-events-A` | Dedicated topic per producer isolates each camera stream, allowing the Spark consumer to subscribe selectively and apply per-stream logic. |
| `camera_id` | `1` | Identifies the source camera for all events published by this producer, enabling traceability in the downstream violation detection logic. |
| `batch_interval` | `5 seconds` | Events are grouped by `batch_id` and published one batch every 5 seconds. It provides a simulation of a camera emitting periodic snapshots, while giving Spark sufficient time to process each micro-batch before the next arrives. |
| `csv_path` | `../data/camera_event_A.csv` | Relative path to the camera event dataset for Producer A, following the submission directory structure defined in the specification. |

### Execution Notes

- The producer runs continuously until manually interrupted (`KeyboardInterrupt`).
- On interrupt or error, `producer_a.close()` is called in the `finally` block to ensure the Kafka
  producer is gracefully shut down and all buffered messages are flushed.
- Run this notebook **concurrently** with Producer B and Producer C notebooks to simulate
  simultaneous multi-camera event ingestion, as required by the streaming join logic in Task 2.1.2.

In [ ]:
from pathlib import Path

from camera_event_producer import CameraEventProducer

HOST_IP = "kafka"  # Docker Compose service name
csv_path = Path("..") / "data" / "camera_event_A.csv"
producer_a = CameraEventProducer(
    bootstrap_servers=[f"{HOST_IP}:9092"],
    topic="camera-events-A",
    camera_id=1,
    csv_path=str(csv_path),
    batch_interval=5
)

try:
    producer_a.publish_batches()
except KeyboardInterrupt:
    print("Producer A stopped by user")
finally:
    producer_a.close()